# 04. Memory-Augmented Navigation: GRU vs Deep Sets (Phase 2)

This notebook trains `GRUMemoryAgent` and `DeepSetsMemoryAgent` and compares them against the **pretrained `GARDENAgent` baseline** (loaded from checkpoint without retraining):
1. **GARDEN Validation Baseline** — load `GARDEN` checkpoint trained in Notebook 03 and evaluate on `val_pairs`.
2. **Sequential Memory (`GRUMemoryAgent`)** — chronological trajectory encoding trained via `RecurrentA2CTrainer` (BPTT).
3. **Set-Based Memory (`DeepSetsMemoryAgent`)** — permutation-invariant visited-node coverage trained via `RecurrentA2CTrainer` (BPTT).
4. **Validation & Test Comparison** — learning curves vs. `GARDEN` reference line and test-set comparison (`R_trunc`, `R_oracle`, `Revisit Rate`, `Ping-Pong`, `Win Rate`).

In [4]:
import os
import pickle
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch as T

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.data.splits import load_splits_and_pairs
from src.env.graph_env import GraphNavEnv
from src.models import DeepSetsMemoryAgent, GARDENAgent, GRUMemoryAgent
from src.training import RecurrentA2CTrainer
from src.utils.evaluator import evaluate_agent
from src.utils.metrics import compute_win_rates

sns.set_theme(style="whitegrid")
DEVICE = "cuda" if T.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Experiment Configuration

In [9]:
GRAPH_NAME = "facebook_1"
SEED = 42
NUM_EPISODES = 500
EVAL_EVERY = 500
VAL_SUBSET_SIZE = 100
TEST_SUBSET_SIZE = 100  # set to None to evaluate on all test_pairs

MEMORY_DIM = 32
STEP_PENALTY = 0.0  # try -0.01 if sparse rewards converge slowly
FREEZE_GAT = False  # set True to freeze pretrained GAT weights from GARDEN_CKPT
GARDEN_CKPT = Path(f"checkpoints/{GRAPH_NAME}/GARDEN_{GRAPH_NAME}_best.pt")

random.seed(SEED)
np.random.seed(SEED)
T.manual_seed(SEED)

graph_dir = Path("data/processed") / GRAPH_NAME
with open(graph_dir / "graph.pkl", "rb") as f:
    g = pickle.load(f)
features = np.load(graph_dir / "features.npy")
splits, val_pairs, test_pairs = load_splits_and_pairs(str(graph_dir))

train_env = GraphNavEnv(g, features, max_steps=100, target_nodes=splits["train"], seed=SEED)
eval_env = GraphNavEnv(g, features, max_steps=100, seed=SEED)
val_sub = val_pairs[:VAL_SUBSET_SIZE]
test_sub = test_pairs[:TEST_SUBSET_SIZE] if TEST_SUBSET_SIZE else test_pairs
print(f"Loaded {GRAPH_NAME}: {g.number_of_nodes()} nodes | val_sub={len(val_sub)} | test_sub={len(test_sub)}")

Loaded facebook_1: 148 nodes | val_sub=100 | test_sub=100


## 2. Validate Pretrained `GARDEN` Baseline (No Retraining)

In [7]:
garden_agent = GARDENAgent(
    env=eval_env,
    hidden_dim=64,
    rep_dim=32,
    lr=1e-4,
    entropy_coef=1e-3,
    use_degree=True,
    device=DEVICE,
    name=f"GARDEN_{GRAPH_NAME}",
)

garden_val_summary = None
if GARDEN_CKPT.exists():
    garden_agent.load_checkpoint(str(GARDEN_CKPT))
    garden_agent.eval()
    garden_val_summary, _ = evaluate_agent(garden_agent, eval_env, val_sub, show_progress=False)
    print(
        f"[GARDEN Validation ({GARDEN_CKPT})] "
        f"R_oracle={garden_val_summary['mean_oracle_ratio']:.2f} | "
        f"R_trunc={garden_val_summary['r_trunc']:.1f}% | "
        f"Revisit={garden_val_summary['mean_revisit_rate']:.2f} | "
        f"Ping-Pong={garden_val_summary['mean_ping_pong_count']:.2f}"
    )
else:
    print(f"[Warning] GARDEN checkpoint not found at {GARDEN_CKPT}. Run Notebook 03 first to train GARDEN.")

[Warning] GARDEN checkpoint not found at checkpoints\facebook_1\GARDEN_facebook_1_best.pt. Run Notebook 03 first to train GARDEN.


## 3. Train `GRUMemoryAgent` and `DeepSetsMemoryAgent` (BPTT)

In [10]:
memory_agents = {
    "GRU_GARDEN": GRUMemoryAgent(
        env=train_env,
        hidden_dim=64,
        rep_dim=32,
        memory_dim=MEMORY_DIM,
        lr=1e-4,
        entropy_coef=1e-3,
        use_degree=True,
        freeze_gat=FREEZE_GAT,
        device=DEVICE,
        name=f"GRU_{GRAPH_NAME}",
    ),
    "DeepSets_GARDEN": DeepSetsMemoryAgent(
        env=train_env,
        hidden_dim=64,
        rep_dim=32,
        memory_dim=MEMORY_DIM,
        lr=1e-4,
        entropy_coef=1e-3,
        use_degree=True,
        freeze_gat=FREEZE_GAT,
        deduplicate=True,
        device=DEVICE,
        name=f"DeepSets_{GRAPH_NAME}",
    ),
}

if FREEZE_GAT and GARDEN_CKPT.exists():
    for label, agent in memory_agents.items():
        agent.load_gat_weights(str(GARDEN_CKPT), freeze=True)
        print(f"Loaded & froze pretrained GAT weights into {label}")

histories = {}
for label, agent in memory_agents.items():
    trainer = RecurrentA2CTrainer(
        agent=agent,
        env=train_env,
        train_targets=splits["train"],
        val_pairs=val_pairs,
        checkpoint_dir=f"checkpoints/{GRAPH_NAME}",
        eval_every=EVAL_EVERY,
        step_penalty=STEP_PENALTY,
    )
    histories[label] = trainer.train(
        num_episodes=NUM_EPISODES,
        val_subset_size=VAL_SUBSET_SIZE,
        verbose=True,
    )

[RecurrentA2CTrainer] Starting training for 500 episodes on GRU_facebook_1...


KeyboardInterrupt: 

## 4. Validation Curves vs. `GARDEN` Reference & Test Set Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for label, hist in histories.items():
    steps_x = [EVAL_EVERY * (i + 1) for i in range(len(hist["val_oracle_ratio"]))]
    axes[0].plot(steps_x, hist["val_oracle_ratio"], marker="o", label=label)
    axes[1].plot(steps_x, hist["val_trunc"], marker="o", label=label)
    axes[2].plot(steps_x, hist["val_revisit_rate"], marker="o", label=label)

if garden_val_summary is not None:
    axes[0].axhline(garden_val_summary["mean_oracle_ratio"], color="black", linestyle="--", label="GARDEN (pretrained)")
    axes[1].axhline(garden_val_summary["r_trunc"], color="black", linestyle="--", label="GARDEN (pretrained)")
    axes[2].axhline(garden_val_summary["mean_revisit_rate"], color="black", linestyle="--", label="GARDEN (pretrained)")

axes[0].set_title(f"Validation R_bar_oracle ({GRAPH_NAME})")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("R_bar_oracle")
axes[0].legend()

axes[1].set_title(f"Validation R_trunc % ({GRAPH_NAME})")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("R_trunc (%)")
axes[1].legend()

axes[2].set_title(f"Validation Revisit Rate ({GRAPH_NAME})")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Revisit Rate")
axes[2].legend()

plt.tight_layout()
plt.show()

# Test evaluation of pretrained GARDEN + best GRU_GARDEN + best DeepSets_GARDEN
eval_agents = {}
if GARDEN_CKPT.exists():
    eval_agents["GARDEN"] = (garden_agent, GARDEN_CKPT)
for label, agent in memory_agents.items():
    eval_agents[label] = (agent, Path(f"checkpoints/{GRAPH_NAME}/{agent.name}_best.pt"))

test_summaries = []
test_records = {}
path_lengths = {}

for label, (agent, ckpt_path) in eval_agents.items():
    if ckpt_path.exists():
        agent.load_checkpoint(str(ckpt_path))
    agent.eval()
    summary, recs = evaluate_agent(agent, eval_env, test_sub, show_progress=False)
    summary["model"] = label
    test_summaries.append(summary)
    test_records[label] = recs
    path_lengths[label] = [r["steps"] for r in recs]

win_rates = compute_win_rates(path_lengths, seed=SEED)
for s in test_summaries:
    s["win_rate"] = win_rates[s["model"]]

df_test = pd.DataFrame(test_summaries)
display(df_test[["model", "r_trunc", "mean_oracle_ratio", "oracle_ratio_ci", "mean_revisit_rate", "mean_ping_pong_count", "win_rate"]])